In [0]:
df = spark.table("post_renewal_churn.cleaned_dataset.final_dataset")
display(df)

In [0]:
print(len(df.columns))

In [0]:
from pyspark.sql.functions import when

df = df.withColumn('prospect_outcome', 
                   when(df['prospect_outcome'] == 'Won', 1)
                   .when(df['prospect_outcome'] == 'Churned', 0)
                   .otherwise(df['prospect_outcome']))

In [0]:
from pyspark.ml.feature import StringIndexer

cols = ["Band", "Connection_Group", "tenure_group", "Anchor_Group"]

for c in cols:
    indexer = StringIndexer(inputCol=c, outputCol=c + "_idx")
    df = indexer.fit(df).transform(df)

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "Serious_Complaint_num",
    when(col("Serious_Complaint") == "Yes", 1).otherwise(0)
)

df = df.withColumn(
    "Other_Complaint_num",
    when(col("Other_Complaint") == "Yes", 1).otherwise(0)
)

df = df.withColumn(
    "Renewal_Impact_Due_to_Price_Increase_num",
    when(col("Renewal_Impact_Due_to_Price_Increase") == "Yes", 1).otherwise(0)
)

df = df.withColumn(
    "Discount_or_Waiver_Requested_num",
    when(col("Discount_or_Waiver_Requested") == "Yes", 1).otherwise(0)
)

df = df.withColumn(
    "Explicit_Competitor_Mention_num",
    when(col("Explicit_Competitor_Mention") == "Yes", 1).otherwise(0)
)

## **CORRELATION MATRIX**

In [0]:
features = [
    "Amount", "Total_Amount", "discount_amount",
    "Gross", "Membership_Net", "Package_Net", "PQQNet",
    "Starting_Gross", "Starting_Membership_Net", "Starting_Package_Net",
    "auto_renewal_score", "tenure_scores", "Tenure_Years",
    "Serious_Complaint_num", "Other_Complaint_num",
    "Renewal_Impact_Due_to_Price_Increase_num",
    "Discount_or_Waiver_Requested_num",
    "Explicit_Competitor_Mention_num",
    "days_to_close",
    
    # Added encoded categorical features
    "Band_idx",
    "Connection_Group_idx",
    "tenure_group_idx",
    "Anchor_Group_idx"
]


In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Assemble vector (exclude target from features)
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_vector = assembler.transform(df.select(*features, "prospect_outcome"))

# Correlation matrix
corr_matrix = Correlation.corr(df_vector, "features").head()[0].toArray()

# Convert to pandas DataFrame for heatmap
corr_df = pd.DataFrame(corr_matrix, columns=features, index=features)

# Plot heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_df, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation Matrix")
plt.show()

# Calculate correlation of each feature with target
assembler_target = VectorAssembler(inputCols=features + ["prospect_outcome"], outputCol="features_target")
df_vector_target = assembler_target.transform(df.select(*features, "prospect_outcome"))
corr_matrix_target = Correlation.corr(df_vector_target, "features_target").head()[0].toArray()

# Get correlations with target (last column)
target_corr = corr_matrix_target[:-1, -1]
top_features_idx = np.argsort(np.abs(target_corr))[::-1][:15]
top_features = [features[i] for i in top_features_idx]
top_corr_values = target_corr[top_features_idx]

# Display top 10 features affecting prospect_outcome
top_corr_df = pd.DataFrame({"Feature": top_features, "Correlation_with_Target": top_corr_values})
display(top_corr_df)

In [0]:
from pyspark.sql.functions import when, col

df = df.withColumn(
    "target",
    when(col("prospect_outcome") == "Won", 1).otherwise(0)
)

## **HYPOTHESIS TESTING AND VALIDATION**

### 1. Days to Close 

In [0]:
# H0: No difference in days_to_close between Won and Churn
# H1: There is a significant difference in days_to_close

import scipy.stats as stats

won = df.filter(df["prospect_outcome"] == 1).select("days_to_close").toPandas()["days_to_close"]
churn = df.filter(df["prospect_outcome"] == 0).select("days_to_close").toPandas()["days_to_close"]

t_stat, p_val = stats.ttest_ind(won, churn)

print("\nDays_to_Close T-stat:", t_stat, "P-value:", p_val, "\n")

if p_val < 0.05:
    print("Reject H0 → Significant difference\n")
else:
    print("Accept H0 → No significant difference\n")

In [0]:
import pandas as pd

plot_df = pd.DataFrame({
    "days_to_close": list(won) + list(churn),
    "Outcome": ["Won"] * len(won) + ["Churned"] * len(churn)
})

plt.figure(figsize=(8,5))

sns.barplot(
    x="Outcome",
    y="days_to_close",
    data=plot_df,
    palette="RdBu",
    ci=95
)

plt.title("Average Days to Close by Outcome", fontsize=14)
plt.xlabel("Prospect Outcome")
plt.ylabel("Average Days to Close")

plt.show()

### 2. Discount Amount

In [0]:

# H0: Discount amount has no effect on outcome
# H1: Discount amount significantly affects outcome

import scipy.stats as stats

won = df.filter(df["prospect_outcome"] == 1).select("discount_amount").toPandas()["discount_amount"]
churn = df.filter(df["prospect_outcome"] == 0).select("discount_amount").toPandas()["discount_amount"]

t_stat, p_val = stats.ttest_ind(won, churn)

print("Discount \n\nT-stat:", t_stat, "P-value:", p_val)

if p_val < 0.05:
    print("Reject H0 → Significant difference\n")
else:
    print("Accept H0 → No significant difference\n")



In [0]:
import pandas as pd

plot_df_discount = pd.DataFrame({
    "discount_amount": list(won) + list(churn),
    "Outcome": ["Won"] * len(won) + ["Churned"] * len(churn)
})

plt.figure(figsize=(8,5))

sns.kdeplot(
    plot_df_discount[plot_df_discount["Outcome"]=="Won"]["discount_amount"],
    label="Won",
    fill=True,
    alpha=0.5
)

sns.kdeplot(
    plot_df_discount[plot_df_discount["Outcome"]=="Churned"]["discount_amount"],
    label="Churned",
    fill=True,
    alpha=0.5
)

plt.title("Discount Distribution: Won vs Churn", fontsize=14)
plt.xlabel("Discount Amount")
plt.ylabel("Density")
plt.legend()

plt.show()

### 3. Serious Complaint

In [0]:
# H0: Serious Complaint and outcome are independent
# H1: Serious Complaint affects outcome

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import ChiSquareTest

assembler = VectorAssembler(inputCols=["Serious_Complaint_num"], outputCol="features")
df_chi = assembler.transform(df)

result = ChiSquareTest.test(df_chi, "features", "prospect_outcome").head()

p_val = result.pValues[0]

print("Serious Complaint → P-value:", p_val)

if p_val < 0.05:
    print("Reject H0 → Significant relationship")
else:
    print("Fail to Reject H0 → No relationship")

In [0]:
import pandas as pd

# Create crosstab from Spark DataFrame
df_pd = df.select("Serious_Complaint_num", "prospect_outcome").toPandas()
ct = pd.crosstab(df_pd["Serious_Complaint_num"], df_pd["prospect_outcome"])

# Normalize (row-wise percentage)
ct_percent = ct.div(ct.sum(axis=1), axis=0)

ct_percent.plot(kind="bar", stacked=True, colormap="YlGnBu", figsize=(8,5))

plt.title(" Distribution of Outcome by Complaint", fontsize=14)
plt.xlabel("Serious Complaint")
plt.ylabel("Percentage")
plt.legend(title="Outcome")

plt.show()

### 4. Auto Renewal Score

In [0]:
# H0: Auto renewal score has no effect on outcome
# H1: Auto renewal score affects outcome

won = df.filter(df["prospect_outcome"] == 1).select("auto_renewal_score").toPandas()["auto_renewal_score"]
churn = df.filter(df["prospect_outcome"] == 0).select("auto_renewal_score").toPandas()["auto_renewal_score"]

t_stat, p_val = stats.ttest_ind(won, churn)

print("Auto Renewal \n \nT-stat:", t_stat, "P-value:", p_val)

if p_val < 0.05:
    print("Reject H0 → Significant difference\n")
else:
    print("Accept H0 → No significant difference\n")

In [0]:
import pandas as pd

plot_df = pd.DataFrame({
    "auto_renewal_score": list(won) + list(churn),
    "Outcome": ["Won"] * len(won) + ["Churn"] * len(churn)
})

plt.figure(figsize=(8,5))

sns.violinplot(
    x="Outcome",
    y="auto_renewal_score",
    data=plot_df,
    palette="Set2",
    inner="quartile"
)

plt.title("Auto Renewal Score by Outcome", fontsize=14)
plt.xlabel("Prospect Outcome")
plt.ylabel("Auto Renewal Score")

plt.show()

### 5. Band

In [0]:
# H0: Band and outcome are independent
# H1: Band affects outcome

assembler = VectorAssembler(inputCols=["Band_idx"], outputCol="features")
df_chi = assembler.transform(df)

result = ChiSquareTest.test(df_chi, "features", "prospect_outcome").head()

p_val = result.pValues[0]

print("Band → P-value:", p_val)

if p_val < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

In [0]:
import seaborn as sns

plt.figure(figsize=(8,5))

sns.heatmap(
    ct,
    annot=True,
    fmt="d",
    cmap="Spectral"
)

plt.title("Heatmap: Band vs Outcome")
plt.xlabel("Outcome")
plt.ylabel("Band")

plt.show()

### 6. Total Amount vs Prospect Outcome

In [0]:
# H0: Total_Amount has no effect on outcome
# H1: Total_Amount significantly affects renewal outcome

import scipy.stats as stats
import pandas as pd

won = df.filter(df["prospect_outcome"] == 1) \
        .select("Total_Amount") \
        .toPandas()["Total_Amount"]

churn = df.filter(df["prospect_outcome"] == 0) \
          .select("Total_Amount") \
          .toPandas()["Total_Amount"]

t_stat, p_val = stats.ttest_ind(won, churn)

print("Total Amount\n")
print("T-stat:", t_stat)
print("P-value:", p_val)

if p_val < 0.05:
    print("Reject H0 → Significant difference")
else:
    print("Fail to Reject H0 → No significant difference")

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

plot_df = pd.DataFrame({
    "Total_Amount": list(won) + list(churn),
    "Outcome": ["Won"] * len(won) + ["Churn"] * len(churn)
})

plt.figure(figsize=(8,5))

sns.barplot(
    x="Outcome",
    y="Total_Amount",
    data=plot_df,
    palette="coolwarm",
    ci=95
)

plt.title("Average Total Amount by Outcome")
plt.show()

### 7. Gross vs Prospect Outcome

In [0]:
# H0: Gross has no effect on prospect_outcome
# H1: Gross significantly affects prospect_outcome

won = df.filter(df["prospect_outcome"] == "1") \
        .select("Gross") \
        .toPandas()["Gross"]

churn = df.filter(df["prospect_outcome"] == "0") \
          .select("Gross") \
          .toPandas()["Gross"]

t_stat, p_val = stats.ttest_ind(won, churn)

print("Gross\n")
print("T-stat:", t_stat)
print("P-value:", p_val)

if p_val < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

In [0]:
plot_df = pd.DataFrame({
    "Gross": list(won) + list(churn),
    "Outcome": ["Won"] * len(won) + ["Churn"] * len(churn)
})

plt.figure(figsize=(8,5))

sns.violinplot(
    x="Outcome",
    y="Gross",
    data=plot_df,
    palette="Set2",
    inner="quartile"
)

plt.title("Gross Distribution by Outcome")
plt.show()

### 8. Payment Method vs Prospect Outcome

In [0]:
# H0: Payment_Method and prospect_outcome are independent
# H1: Payment_Method affects prospect_outcome

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.stat import ChiSquareTest

# Encode Payment_Method if not already encoded
if "Payment_Method_idx" not in df.columns:
    df = StringIndexer(inputCol="Payment_Method", outputCol="Payment_Method_idx").fit(df).transform(df)

assembler = VectorAssembler(inputCols=["Payment_Method_idx"], outputCol="features")
df_chi = assembler.transform(df)

result = ChiSquareTest.test(df_chi, "features", "prospect_outcome").head()
p_val = result.pValues[0]

print("Payment Method → P-value:", p_val)

if p_val < 0.05:
    print("Reject H0 → Significant relationship")
else:
    print("Fail to Reject H0 → No relationship")

In [0]:
import pandas as pd

plot_df = df.select("Payment_Method", "prospect_outcome").toPandas()

ct = pd.crosstab(plot_df["Payment_Method"], plot_df["prospect_outcome"])
ct_percent = ct.div(ct.sum(axis=1), axis=0)

ct_percent.plot(kind="bar", stacked=True, colormap="coolwarm", figsize=(8,5))

plt.title("Outcome Distribution by Payment Method")
plt.ylabel("Percentage")
plt.show()

### 9. Call Direction vs Prospect Outcome


In [0]:
# H0: Call_Direction and prospect_outcome are independent
# H1: Call_Direction affects prospect_outcome

if "Call_Direction_idx" not in df.columns:
    df_chi_input = StringIndexer(inputCol="Call_Direction", outputCol="Call_Direction_idx").fit(df).transform(df)
else:
    df_chi_input = df

assembler = VectorAssembler(inputCols=["Call_Direction_idx"], outputCol="features")
df_chi = assembler.transform(df_chi_input)

result = ChiSquareTest.test(df_chi, "features", "prospect_outcome").head()
p_val = result.pValues[0]

print("Call Direction → P-value:", p_val)

if p_val < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

In [0]:
import seaborn as sns

plot_df = df.select("Call_Direction", "prospect_outcome").toPandas()

plt.figure(figsize=(8,5))

sns.countplot(
    x="Call_Direction",
    hue="prospect_outcome",
    data=plot_df,
    palette="Set2"
)

plt.title("Call Direction vs Outcome")
plt.show()